# Coach Fallback vs Pydantic 对比

这份 notebook 用同一道题、同一条学生回答，直接并排比较两条分析链路：

- `fallback`：`use_ai_reply_analyzer=False`
- `pydantic`：`use_ai_reply_analyzer=True`

重点观察：

- `reply_analysis.source`
- `quality`
- `understands`
- `completed`
- 首轮 `coach` 回复


In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path('/Users/xumuchi/Desktop/TeachAgent')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from coach_agent import FoundryCoachAgent, ErrorType, diagnose_environment

print(json.dumps(diagnose_environment(), ensure_ascii=False, indent=2))

In [ ]:
PROBLEM_TEXT = '已知等差数列 {a_n} 中，a_1=3，a_4=12。求公差 d，并求 a_10 的值。'
STUDENT_PROFILE = '学生知道等差数列公式，但经常不知道先从哪个条件下手。'
EXTRA_RULE = '先引导学生找中间量，再决定是否总结，不要一开始直接给完整答案。'

TEST_REPLIES = [
    '我不会。',
    '是不是先求公差 d？',
    '因为 a_4=a_1+3d，所以 12=3+3d，d=3。',
]

print('测试题目：', PROBLEM_TEXT)
print('测试学生画像：', STUDENT_PROFILE)
print('测试回答条数：', len(TEST_REPLIES))

In [ ]:
def build_session(agent):
    return agent.create_session(
        problem_text=PROBLEM_TEXT,
        error_type=ErrorType.MISSING_STRATEGY,
        student_profile=STUDENT_PROFILE,
        extra_rule=EXTRA_RULE,
        max_turns=4,
    )


def pack_result(label, result):
    return {
        'label': label,
        'analysis_source': result.reply_analysis.source,
        'quality': result.reply_quality.value,
        'understands': result.reply_analysis.understands,
        'completed': result.reply_analysis.completed,
        'reason': result.reply_analysis.reason,
        'strategy_mode': result.strategy.mode.value,
        'strategy_prompt': result.strategy.prompt,
        'coach_reply': result.content,
    }


def run_compare(student_reply):
    fallback_agent = FoundryCoachAgent(use_ai_reply_analyzer=False)
    fallback_session = build_session(fallback_agent)
    fallback_result = fallback_agent.reply(student_reply, session=fallback_session)

    pydantic_agent = FoundryCoachAgent(use_ai_reply_analyzer=True)
    pydantic_session = build_session(pydantic_agent)
    pydantic_result = pydantic_agent.reply(student_reply, session=pydantic_session)

    return {
        'student_reply': student_reply,
        'fallback': pack_result('fallback', fallback_result),
        'pydantic': pack_result('pydantic', pydantic_result),
    }


def show_compare(payload):
    print('\n' + '=' * 80)
    print('学生回答：', payload['student_reply'])
    print('=' * 80)
    print('\n【Fallback】')
    print(json.dumps(payload['fallback'], ensure_ascii=False, indent=2))
    print('\n【Pydantic】')
    print(json.dumps(payload['pydantic'], ensure_ascii=False, indent=2))

In [ ]:
records = []
for reply in TEST_REPLIES:
    record = run_compare(reply)
    records.append(record)
    show_compare(record)

In [ ]:
summary_rows = []
for record in records:
    summary_rows.append({
        'student_reply': record['student_reply'],
        'fallback_source': record['fallback']['analysis_source'],
        'fallback_quality': record['fallback']['quality'],
        'fallback_understands': record['fallback']['understands'],
        'fallback_completed': record['fallback']['completed'],
        'pydantic_source': record['pydantic']['analysis_source'],
        'pydantic_quality': record['pydantic']['quality'],
        'pydantic_understands': record['pydantic']['understands'],
        'pydantic_completed': record['pydantic']['completed'],
    })

print(json.dumps(summary_rows, ensure_ascii=False, indent=2))

In [ ]:
for record in records:
    assert record['fallback']['analysis_source'] == 'fallback_heuristic', record['fallback']['analysis_source']
    assert record['pydantic']['analysis_source'] == 'ai_tool_pydantic_parse', record['pydantic']['analysis_source']

print('OK: fallback 与 pydantic 两条链路都按预期跑通。')